# EXP-QRRFULL-TinyMLP2-XJTU-10SEEDS-SAME-CONFIG

**Goal:** run **QRR_Full + TinyMLP2** on all six XJTU batches with the same recurrent QRR architecture and TinyMLP2 readout settings used in the good TJU TinyMLP2 run.

Architecture:
- **QRR_Full**: 10 qubits = 4 recurrent memory qubits + 6 processor qubits
- Coupling: `c = 0.3`, direct/reverse/skip couplings as in the original QRR_Full notebooks
- Recurrent memory update: `memory_angles = (memory_readout + 1.0) * pi`
- Reservoir features: 57

Readout:
- **TinyMLP2**: `57 -> 2 -> 2 -> 1`
- Parameters: **125**
- Dropout: `0.0`
- Epochs/patience: `300 / 40`

Protocol:
- XJTU leave-one-cell-out cross-validation inside each batch
- Training normalization is fitted only on training cells
- Default setting regenerates QRR_Full features from XJTU data and saves them in a fresh cache.


Ten-seed extension:
- QRR reservoir remains fixed with `SEED = 42`.
- TinyMLP2 is trained using readout seeds `0` through `9`.
- QRR features are generated once per LOO fold and reused across all ten readout seeds.


In [1]:

# ═══════════════════════════════════════════════════════════════════
# Setup
# ═══════════════════════════════════════════════════════════════════
!pip -q install pennylane

from google.colab import drive
drive.mount('/content/drive')

import os, time, json, warnings, pickle, datetime, hashlib, copy
from pathlib import Path
import numpy as np
import scipy.io as sio
import pandas as pd
import pennylane as qml

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

PROJECT_ROOT = "/content/drive/MyDrive/HB/QRC_Battery_SOH"
# Remount Google Drive if the connection is lost
if not os.path.exists(PROJECT_ROOT):
    print("Google Drive not mounted correctly, re-mounting...")
    drive.mount('/content/drive', force_remount=True)
    os.chdir(PROJECT_ROOT)
else:
    os.chdir(PROJECT_ROOT)

XJTU_PATH = os.path.join(PROJECT_ROOT, "data/raw/XJITU")

EXP_ID = "EXP-QRRFULL-TinyMLP2-XJTU-10SEEDS-SAME-CONFIG"
CKPT_DIR = f"checkpoints/{EXP_ID}"
RES_DIR  = f"results/{EXP_ID}"
LOG_DIR  = f"logs/{EXP_ID}"
FEAT_DIR = f"features/{EXP_ID}"
for d in [CKPT_DIR, RES_DIR, LOG_DIR, FEAT_DIR]:
    os.makedirs(d, exist_ok=True)

log_path = f"{LOG_DIR}/{EXP_ID}_log.txt"
def log(msg):
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(log_path, "a") as f:
        f.write(line + "\n")

log(f"Experiment: {EXP_ID}")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("XJTU_PATH:", XJTU_PATH)
print("Exists:", os.path.exists(XJTU_PATH))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 39.3 MB/s eta 0:00:00
Mounted at /content/drive
[06:13:56] Experiment: EXP-QRRFULL-TinyMLP2-XJTU-10SEEDS-SAME-CONFIG
PROJECT_ROOT: /content/drive/MyDrive/HB/QRC_Battery_SOH
XJTU_PATH: /content/drive/MyDrive/HB/QRC_Battery_SOH/data/raw/XJITU
Exists: True


In [2]:

# ═══════════════════════════════════════════════════════════════════
# QRR_Full 10q recurrent reservoir class
# - 4 recurrent memory qubits
# - 6 processor qubits
# - original cross-register coupling strength
# - recurrent memory update ENABLED
# ═══════════════════════════════════════════════════════════════════
class QuantumRecurrentReservoir10q:
    def __init__(self, n_memory=4, n_processor=6, n_reservoir_layers=2, n_inputs=8, seed=42,
                 coupling_strength=0.3):
        self.n_memory = n_memory
        self.n_processor = n_processor
        self.n_qubits = n_memory + n_processor
        self.n_reservoir_layers = n_reservoir_layers
        self.n_inputs = n_inputs
        self.seed = seed
        self.coupling_strength = coupling_strength
        self.memory_wires = list(range(n_memory))
        self.proc_wires = list(range(n_memory, self.n_qubits))
        self.proc_pairs = [(i, j) for i in self.proc_wires for j in self.proc_wires if i < j]
        self.cross_pairs = [(i, j) for i in self.memory_wires for j in self.proc_wires]
        self.n_single = n_processor * 3
        self.n_proc_pairs = len(self.proc_pairs)
        self.n_cross_pairs = len(self.cross_pairs)
        self.n_outputs = self.n_single + self.n_proc_pairs + self.n_cross_pairs  # 57

        rng = np.random.RandomState(seed)
        self.reservoir_params = rng.uniform(0, 2*np.pi, size=(n_reservoir_layers, self.n_qubits, 2))
        self.input_projection = rng.randn(n_inputs, n_processor) * 0.5
        self.memory_rotation = rng.uniform(0, 2*np.pi, size=(n_memory, 2))
        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        self.memory_angles = np.zeros(n_memory)
        self._build_circuit()

    def _build_circuit(self):
        n_q, n_m, n_p, n_l = self.n_qubits, self.n_memory, self.n_processor, self.n_reservoir_layers
        rp, mw, pw = self.reservoir_params, self.memory_wires, self.proc_wires
        pp, cp, c, mr = self.proc_pairs, self.cross_pairs, self.coupling_strength, self.memory_rotation

        @qml.qnode(self.dev)
        def circuit(pi, ma):
            # Memory register initialized from the recurrent memory angles, plus fixed rotations.
            for i, w in enumerate(mw):
                qml.RY(ma[i], wires=w)
                qml.RZ(mr[i, 0], wires=w)

            # Processor input encoding.
            for i, w in enumerate(pw):
                if i < len(pi):
                    qml.RY(pi[i], wires=w)
                    qml.RZ(pi[i] * 0.5, wires=w)

            # Within-register entanglement.
            for i in range(n_m - 1):
                qml.CNOT(wires=[mw[i], mw[i+1]])
            for i in range(n_p - 1):
                qml.CNOT(wires=[pw[i], pw[i+1]])
            qml.CNOT(wires=[pw[-1], pw[0]])

            # Cross-register memory-processor coupling: direct, reverse, skip.
            for i in range(n_m):
                qml.CRY(c * np.pi, wires=[mw[i], pw[i]])
                qml.CRY(c * 0.5 * np.pi, wires=[pw[i], mw[i]])
            for i in range(n_m):
                qml.CRY(c * 0.3 * np.pi, wires=[mw[i], pw[(i+2) % n_p]])

            # Fixed reservoir layers.
            for l in range(n_l):
                for i in range(n_q):
                    qml.RY(rp[l, i, 0], wires=i)
                    qml.RZ(rp[l, i, 1], wires=i)
                for i in range(n_q):
                    qml.CNOT(wires=[i, (i+1) % n_q])
                for i in range(0, n_q - 2, 2):
                    qml.CNOT(wires=[i, i+2])

            obs = []
            for w in pw:
                obs.extend([
                    qml.expval(qml.PauliX(w)),
                    qml.expval(qml.PauliY(w)),
                    qml.expval(qml.PauliZ(w)),
                ])
            for (i, j) in pp:
                obs.append(qml.expval(qml.PauliZ(i) @ qml.PauliZ(j)))
            for (i, j) in cp:
                obs.append(qml.expval(qml.PauliZ(i) @ qml.PauliZ(j)))
            # extra memory Z outputs are used to update recurrent memory angles
            for w in mw:
                obs.append(qml.expval(qml.PauliZ(w)))
            return obs

        self._circuit = circuit

    def reset(self):
        self.memory_angles = np.zeros(self.n_memory)

    def process_single(self, x):
        pi = np.clip(x @ self.input_projection, -np.pi, np.pi)
        ao = np.array(self._circuit(pi, self.memory_angles))
        features = ao[:self.n_outputs]
        memory_readout = ao[self.n_outputs:]
        # QRR_Full: recurrent memory update, same as original QRR + MLP notebook.
        self.memory_angles = (memory_readout + 1.0) * np.pi
        return features.astype(np.float32)

    def process_sequence(self, X, verbose=True):
        self.reset()
        n = X.shape[0]
        states = np.zeros((n, self.n_outputs), dtype=np.float32)
        t0 = time.time()
        for t in range(n):
            states[t] = self.process_single(X[t]).astype(np.float32)
            if verbose and (t + 1) % 500 == 0:
                el = time.time() - t0
                print(f"      {t+1}/{n} ({el:.0f}s, ~{el/(t+1)*(n-t-1):.0f}s left)", end="\n")
        if verbose:
            print(f"      {n}/{n} done in {time.time()-t0:.1f}s                        ")
        return states

class PhysicsConstrainedReadout:
    def __init__(self, n_features=57, alpha=1.0, mono_weight=0.1):
        self.alpha = alpha
        self.mono_weight = mono_weight
        self.weights = self.bias = self.feature_mean = self.feature_std = None
        self.n_trainable = None

    def _normalize(self, R, fit=False):
        if fit:
            self.feature_mean = R.mean(axis=0)
            self.feature_std = R.std(axis=0) + 1e-8
        return (R - self.feature_mean) / self.feature_std

    def fit(self, R, y):
        R_n = self._normalize(R, fit=True)
        n, d = R_n.shape
        R_b = np.hstack([R_n, np.ones((n, 1))])
        reg = self.alpha * np.eye(R_b.shape[1])
        reg[-1, -1] = 0
        if self.mono_weight > 0 and n > 1:
            diff = R_b[1:] - R_b[:-1]
            reg += self.mono_weight * (diff.T @ diff)
        w = np.linalg.solve(R_b.T @ R_b + reg, R_b.T @ y)
        self.weights, self.bias = w[:-1], w[-1]
        self.n_trainable = len(w)
        return self

    def predict(self, R):
        R_n = self._normalize(R, fit=False)
        return R_n @ self.weights + self.bias


def compute_metrics(y_true, y_pred):
    y_pred = np.asarray(y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-8, None)))
    r2 = r2_score(y_true, y_pred)
    return {"RMSE": float(rmse), "MAE": float(mae), "MAPE": float(mape), "R2": float(r2), "Acc": float((1-mape)*100)}


def normalize_features(features, fit=False, stats=None):
    if fit or stats is None:
        stats = {"mean": features.mean(axis=0), "std": features.std(axis=0) + 1e-8}
    normalized = (features - stats["mean"]) / stats["std"]
    normalized = np.clip(normalized, -3, 3) * (np.pi / 3)
    return normalized, stats

print(f"✅ QRR_Full outputs: {QuantumRecurrentReservoir10q(n_inputs=8).n_outputs} features")

✅ QRR_Full outputs: 57 features


In [3]:

# ═══════════════════════════════════════════════════════════════════
# XJTU loader
# ═══════════════════════════════════════════════════════════════════
def load_xjtu_cell(filepath):
    mat = sio.loadmat(filepath)
    raw = mat["data"]
    summary = mat["summary"]
    n_cycles = raw.shape[1]

    def get_sf(name):
        val = summary[name][0, 0]
        return val[0, 0].flatten() if val.dtype == object else val.flatten()

    disch_cap = get_sf("discharge_capacity_Ah")
    disch_mean_v = get_sf("discharge_mean_voltage")
    disch_median_v = get_sf("discharge_median_voltage")
    initial_cap = disch_cap[0]
    soh = disch_cap / initial_cap

    features = np.zeros((n_cycles, 8), dtype=np.float32)
    for i in range(n_cycles):
        features[i, 0] = disch_mean_v[i]
        features[i, 1] = disch_median_v[i]
        features[i, 5] = disch_cap[i]
        try:
            v = raw["voltage_V"][0, i].flatten()
            curr = raw["current_A"][0, i].flatten()
            temp = raw["temperature_C"][0, i].flatten()
            t_min = raw["relative_time_min"][0, i].flatten()
            features[i, 2] = t_min[-1] / 60.0
            features[i, 3] = np.mean(temp)
            features[i, 4] = np.max(temp)
            dm = curr < -0.01
            if np.sum(dm) > 10:
                vd, td = v[dm], t_min[dm]
                if (td[-1] - td[0]) > 0:
                    features[i, 6] = np.polyfit(td - td[0], vd, 1)[0]
            dur = t_min[-1] / 60.0
            if dur > 0:
                features[i, 7] = (temp[-1] - temp[0]) / dur
        except Exception:
            pass
    return features.astype(np.float32), soh.astype(np.float32), float(initial_cap), int(n_cycles)

xjtu_dataset = {}
xjtu_batches = {}

batches = sorted([d for d in os.listdir(XJTU_PATH)
                  if os.path.isdir(os.path.join(XJTU_PATH, d)) and "Batch" in d])
print("Available batches:", batches)

for bn in batches:
    bp = os.path.join(XJTU_PATH, bn)
    mf = sorted([f for f in os.listdir(bp) if f.endswith('.mat')])
    bc = []
    for m in mf:
        cid = f"{bn}_{m.replace('.mat', '')}"
        try:
            f, s, ic, nc = load_xjtu_cell(os.path.join(bp, m))
            xjtu_dataset[cid] = {"features": f, "soh": s, "n_cycles": nc, "batch": bn, "initial_capacity": ic}
            bc.append(cid)
        except Exception as e:
            print("Failed:", bn, m, e)
    xjtu_batches[bn] = bc
    print(f"  {bn}: {len(bc)} cells")
print(f"✅ XJTU: {len(xjtu_dataset)} cells loaded")


Available batches: ['Batch-1', 'Batch-2', 'Batch-3', 'Batch-4', 'Batch-5', 'Batch-6']
  Batch-1: 8 cells
  Batch-2: 15 cells
  Batch-3: 8 cells
  Batch-4: 8 cells
  Batch-5: 8 cells
  Batch-6: 8 cells
✅ XJTU: 55 cells loaded


In [4]:

# ═══════════════════════════════════════════════════════════════════
# Configuration — QRR_Full + TinyMLP2
# ═══════════════════════════════════════════════════════════════════
N_INPUTS = 8
WARMUP = 5
SEED = 42
SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  # readout/validation seeds; QRR reservoir remains fixed by SEED=42

MODEL_NAME = "QRR_Full"
COUPLING_STRENGTH = 0.3

# Run all six XJTU batches by default. Use e.g. ["Batch-6"] for quick debug.
BATCHES_TO_RUN = None

# Fresh core-QRR recomputation by default.
# If you want to reuse features later, set REGENERATE_FEATURES=False.
USE_FEATURE_CACHE = False
REGENERATE_FEATURES = True

RUN_RIDGE = False
RUN_MLP = True

# TinyMLP2 settings — matched to the good TJU TinyMLP2 run: Tanh, dropout=0.0, epochs=300, patience=40.
MLP_HIDDEN_DIMS = (2, 2)       # 57 -> 2 -> 2 -> 1
MLP_DROPOUT = 0.0
MLP_EPOCHS = 300
MLP_LR = 0.001
MLP_BATCH_SIZE = 64
MLP_PATIENCE = 40
MLP_VAL_SPLIT = 0.2
MLP_WEIGHT_DECAY = 1e-5

def count_mlp_params(n_inputs, hidden_dims):
    dims = [n_inputs] + list(hidden_dims) + [1]
    return sum(dims[i] * dims[i+1] + dims[i+1] for i in range(len(dims)-1))

MLP_PARAMS = count_mlp_params(57, MLP_HIDDEN_DIMS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

REF_XJTU = {
    "QPINN": {
        "Batch-1": 99.74, "Batch-2": 99.51, "Batch-3": 99.39,
        "Batch-4": 99.61, "Batch-5": 99.56, "Batch-6": 99.31,
        "Avg": 99.52,
    },
    "QRR_Full_Ridge": {
        "Batch-1": 98.91, "Batch-2": 98.11, "Batch-3": 99.01,
        "Batch-4": 98.02, "Batch-5": 96.35, "Batch-6": 89.36,
        "Avg": 96.63,
    },
    "CQRF-c0.3_Ridge": {
        "Batch-1": 98.96, "Batch-2": 98.11, "Batch-3": 98.93,
        "Batch-4": 98.02, "Batch-5": 96.49, "Batch-6": 90.62,
        "Avg": 96.86,
    },
    "QRR_Full_MLP_2401p": {
        "Batch-1": 99.06, "Batch-2": 98.52, "Batch-3": 99.47,
        "Batch-4": 98.48, "Batch-5": 96.79, "Batch-6": 96.22,
        "Avg": 98.09,
    },
    "CQRF-c0.3_MLP": {
        "Batch-1": 97.22, "Batch-2": 95.63, "Batch-3": 98.33,
        "Batch-4": 97.51, "Batch-5": 92.64, "Batch-6": 97.77,
        "Avg": 96.52,
    },
}

print("MODEL_NAME:", MODEL_NAME)
print("COUPLING_STRENGTH:", COUPLING_STRENGTH)
print("RUN_RIDGE:", RUN_RIDGE, "RUN_MLP:", RUN_MLP)
print("WARMUP:", WARMUP, "N_INPUTS:", N_INPUTS)
print("BATCHES_TO_RUN:", BATCHES_TO_RUN if BATCHES_TO_RUN is not None else "ALL")
print("REGENERATE_FEATURES:", REGENERATE_FEATURES)
print("MLP architecture:", [57] + list(MLP_HIDDEN_DIMS) + [1], f"dropout={MLP_DROPOUT}")
print("MLP_PARAMS:", MLP_PARAMS, "device:", device)


MODEL_NAME: QRR_Full
COUPLING_STRENGTH: 0.3
RUN_RIDGE: False RUN_MLP: True
WARMUP: 5 N_INPUTS: 8
BATCHES_TO_RUN: ALL
REGENERATE_FEATURES: True
MLP architecture: [57, 2, 2, 1] dropout=0.0
MLP_PARAMS: 125 device: cpu


In [5]:
# ═══════════════════════════════════════════════════════════════════
# Fold-safe QRR_Full feature generation + 10-seed TinyMLP2 readout
# ═══════════════════════════════════════════════════════════════════
def safe_name(s):
    return ''.join(ch if ch.isalnum() or ch in ['-', '_'] else '_' for ch in str(s))


def fold_cache_path(model_name, batch_name, test_cell):
    return os.path.join(
        FEAT_DIR,
        f"{safe_name(model_name)}_{safe_name(batch_name)}_{safe_name(test_cell)}.npz",
    )


def build_train_test_arrays(train_cells, test_cell):
    Xtr = np.vstack([xjtu_dataset[c]["features"] for c in train_cells])
    ytr = np.concatenate([xjtu_dataset[c]["soh"] for c in train_cells])
    Xte = xjtu_dataset[test_cell]["features"]
    yte = xjtu_dataset[test_cell]["soh"]
    return Xtr, ytr, Xte, yte


def apply_warmup(X, y):
    if len(X) > WARMUP:
        return X[WARMUP:], y[WARMUP:]
    return X, y


def generate_or_load_qrrfull_features(batch_name, test_cell, train_cells):
    cache_file = fold_cache_path(MODEL_NAME, batch_name, test_cell)
    if USE_FEATURE_CACHE and (not REGENERATE_FEATURES) and os.path.exists(cache_file):
        z = np.load(cache_file, allow_pickle=True)
        return z["Rtr"], z["ytr"], z["Rte"], z["yte"]

    Xtr, ytr, Xte, yte = build_train_test_arrays(train_cells, test_cell)

    # Original training-only normalization.
    Xtr_n, stats = normalize_features(Xtr, fit=True)
    Xte_n, _ = normalize_features(Xte, fit=False, stats=stats)

    Xtr_n, ytr = apply_warmup(Xtr_n, ytr)
    Xte_n, yte = apply_warmup(Xte_n, yte)

    # QRR reservoir configuration is unchanged and fixed with SEED=42.
    qrr = QuantumRecurrentReservoir10q(
        n_memory=4,
        n_processor=6,
        n_reservoir_layers=2,
        n_inputs=N_INPUTS,
        seed=SEED,
        coupling_strength=COUPLING_STRENGTH,
    )

    log(f"      Generating QRR_Full features: train={len(ytr)}, test={len(yte)}")
    Rtr = qrr.process_sequence(Xtr_n, verbose=True).astype(np.float32)

    # Preserve the original uploaded XJTU QRR protocol exactly:
    # reset, replay the training sequence, then process the test sequence.
    qrr.reset()
    _ = qrr.process_sequence(Xtr_n, verbose=False)
    Rte = qrr.process_sequence(Xte_n, verbose=False).astype(np.float32)

    if USE_FEATURE_CACHE:
        np.savez_compressed(
            cache_file,
            Rtr=Rtr.astype(np.float32),
            ytr=ytr.astype(np.float32),
            Rte=Rte.astype(np.float32),
            yte=yte.astype(np.float32),
            train_cells=np.array(train_cells, dtype=object),
            test_cell=np.array(test_cell, dtype=object),
            batch_name=np.array(batch_name, dtype=object),
            warmup=np.array(WARMUP),
            model_name=np.array(MODEL_NAME, dtype=object),
            reservoir_seed=np.array(SEED),
        )
    return Rtr, ytr, Rte, yte


class TinyMLP2Readout(nn.Module):
    def __init__(self, n_inputs=57, hidden_dims=(2, 2), dropout=0.0):
        super().__init__()
        layers = []
        prev = n_inputs
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.Tanh())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.n_trainable = sum(
            p.numel() for p in self.parameters() if p.requires_grad
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_tinymlp2_readout(R_train, y_train, R_test, seed):
    """
    The architecture and training hyperparameters are unchanged.

    Only these stochastic readout components use `seed`:
      - training/validation permutation,
      - TinyMLP2 initialization,
      - mini-batch shuffle order.
    """
    R_train = R_train.astype(np.float32)
    y_train = y_train.astype(np.float32)
    R_test = R_test.astype(np.float32)

    feat_mean = R_train.mean(axis=0)
    feat_std = R_train.std(axis=0) + 1e-8
    R_train_norm = (R_train - feat_mean) / feat_std
    R_test_norm = (R_test - feat_mean) / feat_std

    n = len(R_train_norm)
    n_val = max(1, int(n * MLP_VAL_SPLIT))
    indices = np.random.RandomState(seed).permutation(n)
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]

    X_tr = torch.FloatTensor(R_train_norm[train_idx]).to(device)
    y_tr = torch.FloatTensor(y_train[train_idx]).to(device)
    X_val = torch.FloatTensor(R_train_norm[val_idx]).to(device)
    y_val = torch.FloatTensor(y_train[val_idx]).to(device)
    X_te = torch.FloatTensor(R_test_norm).to(device)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    loader_generator = torch.Generator()
    loader_generator.manual_seed(seed)
    train_loader = DataLoader(
        TensorDataset(X_tr, y_tr),
        batch_size=MLP_BATCH_SIZE,
        shuffle=True,
        generator=loader_generator,
    )

    model = TinyMLP2Readout(
        n_inputs=R_train.shape[1],
        hidden_dims=MLP_HIDDEN_DIMS,
        dropout=MLP_DROPOUT,
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=MLP_LR,
        weight_decay=MLP_WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        patience=10,
        factor=0.5,
        min_lr=1e-6,
    )
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "lr": []}

    for epoch in range(MLP_EPOCHS):
        model.train()
        losses = []
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val), y_val).item()
        scheduler.step(val_loss)

        history["train_loss"].append(
            float(np.mean(losses)) if losses else float("nan")
        )
        history["val_loss"].append(float(val_loss))
        history["lr"].append(float(optimizer.param_groups[0]["lr"]))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= MLP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        y_pred = model(X_te).cpu().numpy()

    best = {
        "valid_loss": float(best_val_loss),
        "n_iter": epoch + 1,
        "params": int(model.n_trainable),
        "history": history,
    }
    return y_pred, best


selected_batches = batches if BATCHES_TO_RUN is None else BATCHES_TO_RUN

results = {
    "experiment": EXP_ID,
    "dataset": "XJTU",
    "model": MODEL_NAME,
    "readout": "TinyMLP2",
    "reservoir_seed": SEED,
    "readout_seeds": SEEDS,
    "QRRFull_TinyMLP2": {
        "by_seed": {str(seed): {} for seed in SEEDS},
        "aggregate": {},
    },
}

fold_rows = []
seed_summary_rows = []

t_global = time.time()
log(
    f"\n{'='*100}\n"
    "  XJTU — QRR_Full features + TinyMLP2 readout — 10 seeds\n"
    f"{'='*100}"
)
log(f"  Fixed QRR reservoir seed: {SEED}")
log(f"  TinyMLP2 readout seeds: {SEEDS}")

for bn in selected_batches:
    batch_cells = sorted(xjtu_batches.get(bn, []))
    if len(batch_cells) < 2:
        log(f"⚠️ Skipping {bn}: {len(batch_cells)} cells")
        continue

    log(
        f"\n{'-'*70}\n"
        f"  {bn}: {len(batch_cells)} LOO folds × {len(SEEDS)} readout seeds\n"
        f"{'-'*70}"
    )
    batch_metrics_by_seed = {seed: [] for seed in SEEDS}
    t_batch = time.time()

    for fi, test_cell in enumerate(batch_cells):
        train_cells = [c for c in batch_cells if c != test_cell]
        t_fold = time.time()
        log(f"    Fold {fi+1}/{len(batch_cells)}: test={test_cell}")

        # Generate the expensive recurrent QRR features only once per fold.
        Rtr, ytr, Rte, yte = generate_or_load_qrrfull_features(
            bn,
            test_cell,
            train_cells,
        )
        feature_time_sec = time.time() - t_fold

        if RUN_MLP:
            for seed in SEEDS:
                t_seed = time.time()
                yp, best = train_tinymlp2_readout(
                    Rtr,
                    ytr,
                    Rte,
                    seed=seed,
                )
                readout_time_sec = time.time() - t_seed
                m = compute_metrics(yte, yp)
                batch_metrics_by_seed[seed].append(m)

                fold_rows.append({
                    "batch": bn,
                    "seed": seed,
                    "fold": fi + 1,
                    "test_cell": test_cell,
                    "readout": "TinyMLP2",
                    "reservoir_seed": SEED,
                    "valid_loss": best["valid_loss"],
                    "params": best["params"],
                    "n_iter": best["n_iter"],
                    "feature_time_sec": round(feature_time_sec, 2),
                    "readout_time_sec": round(readout_time_sec, 2),
                    **m,
                })

                log(
                    f"      Seed {seed:02d} → "
                    f"MAPE={m['MAPE']:.6f} "
                    f"RMSE={m['RMSE']:.6f} "
                    f"Acc={m['Acc']:.2f}% "
                    f"params={best['params']} "
                    f"iter={best['n_iter']} "
                    f"val_loss={best['valid_loss']:.6f}"
                )

        if (fi + 1) % 5 == 0 or (fi + 1) == len(batch_cells):
            completed_metrics = [
                metric
                for seed in SEEDS
                for metric in batch_metrics_by_seed[seed]
            ]
            running_acc = (
                np.mean([x["Acc"] for x in completed_metrics])
                if completed_metrics
                else float("nan")
            )
            log(
                f"    Progress {fi+1}/{len(batch_cells)} | "
                f"running 10-seed mean Acc={running_acc:.2f}%"
            )

    # One LOO batch result for each readout seed.
    for seed in SEEDS:
        seed_metrics = batch_metrics_by_seed[seed]
        if not seed_metrics:
            continue

        seed_mape = float(np.mean([m["MAPE"] for m in seed_metrics]))
        seed_rmse = float(np.mean([m["RMSE"] for m in seed_metrics]))
        seed_mae = float(np.mean([m["MAE"] for m in seed_metrics]))
        seed_r2 = float(np.mean([m["R2"] for m in seed_metrics]))
        seed_acc = float((1 - seed_mape) * 100)

        seed_result = {
            "accuracy": round(seed_acc, 6),
            "MAPE": round(seed_mape, 8),
            "RMSE": round(seed_rmse, 8),
            "MAE": round(seed_mae, 8),
            "R2": round(seed_r2, 8),
            "n_folds": len(seed_metrics),
        }
        results["QRRFull_TinyMLP2"]["by_seed"][str(seed)][bn] = seed_result

        seed_summary_rows.append({
            "Batch": bn,
            "seed": seed,
            "reservoir_seed": SEED,
            **seed_result,
        })

        log(
            f"  → {bn}/seed={seed:02d}: "
            f"MAPE={seed_mape:.6f} "
            f"RMSE={seed_rmse:.6f} "
            f"Acc={seed_acc:.2f}%"
        )

    # Mean ± sample standard deviation across the ten seed-level batch values.
    batch_seed_results = [
        results["QRRFull_TinyMLP2"]["by_seed"][str(seed)][bn]
        for seed in SEEDS
        if bn in results["QRRFull_TinyMLP2"]["by_seed"][str(seed)]
    ]

    if batch_seed_results:
        acc_values = np.array(
            [x["accuracy"] for x in batch_seed_results],
            dtype=float,
        )
        mape_values = np.array(
            [x["MAPE"] for x in batch_seed_results],
            dtype=float,
        )
        rmse_values = np.array(
            [x["RMSE"] for x in batch_seed_results],
            dtype=float,
        )
        mae_values = np.array(
            [x["MAE"] for x in batch_seed_results],
            dtype=float,
        )
        r2_values = np.array(
            [x["R2"] for x in batch_seed_results],
            dtype=float,
        )

        ddof = 1 if len(batch_seed_results) > 1 else 0
        aggregate_result = {
            "accuracy_mean": round(float(acc_values.mean()), 6),
            "accuracy_std": round(float(acc_values.std(ddof=ddof)), 6),
            "MAPE_mean": round(float(mape_values.mean()), 8),
            "MAPE_std": round(float(mape_values.std(ddof=ddof)), 8),
            "RMSE_mean": round(float(rmse_values.mean()), 8),
            "RMSE_std": round(float(rmse_values.std(ddof=ddof)), 8),
            "MAE_mean": round(float(mae_values.mean()), 8),
            "MAE_std": round(float(mae_values.std(ddof=ddof)), 8),
            "R2_mean": round(float(r2_values.mean()), 8),
            "R2_std": round(float(r2_values.std(ddof=ddof)), 8),
            "n_seeds": len(batch_seed_results),
            "n_folds_per_seed": len(batch_cells),
        }
        results["QRRFull_TinyMLP2"]["aggregate"][bn] = aggregate_result

        log(
            f"  ⇒ {bn}/10-seed aggregate: "
            f"MAPE={aggregate_result['MAPE_mean']:.6f}"
            f"±{aggregate_result['MAPE_std']:.6f} "
            f"RMSE={aggregate_result['RMSE_mean']:.6f}"
            f"±{aggregate_result['RMSE_std']:.6f} "
            f"Acc={aggregate_result['accuracy_mean']:.2f}"
            f"±{aggregate_result['accuracy_std']:.2f}%"
        )

    log(f"  ✅ {bn} completed in {time.time()-t_batch:.0f}s")


# Six-batch macro result for each seed.
for seed in SEEDS:
    seed_key = str(seed)
    seed_batches = [
        results["QRRFull_TinyMLP2"]["by_seed"][seed_key][bn]
        for bn in selected_batches
        if bn in results["QRRFull_TinyMLP2"]["by_seed"][seed_key]
    ]

    if not seed_batches:
        continue

    macro_mape = float(np.mean([x["MAPE"] for x in seed_batches]))
    macro_rmse = float(np.mean([x["RMSE"] for x in seed_batches]))
    macro_mae = float(np.mean([x["MAE"] for x in seed_batches]))
    macro_r2 = float(np.mean([x["R2"] for x in seed_batches]))
    macro_acc = float((1 - macro_mape) * 100)

    macro_result = {
        "accuracy": round(macro_acc, 6),
        "MAPE": round(macro_mape, 8),
        "RMSE": round(macro_rmse, 8),
        "MAE": round(macro_mae, 8),
        "R2": round(macro_r2, 8),
        "n_batches": len(seed_batches),
    }
    results["QRRFull_TinyMLP2"]["by_seed"][seed_key]["Avg"] = macro_result

    seed_summary_rows.append({
        "Batch": "Avg",
        "seed": seed,
        "reservoir_seed": SEED,
        **macro_result,
    })


# Mean ± sample standard deviation across the ten macro values.
macro_seed_results = [
    results["QRRFull_TinyMLP2"]["by_seed"][str(seed)]["Avg"]
    for seed in SEEDS
    if "Avg" in results["QRRFull_TinyMLP2"]["by_seed"][str(seed)]
]

if macro_seed_results:
    acc_values = np.array(
        [x["accuracy"] for x in macro_seed_results],
        dtype=float,
    )
    mape_values = np.array(
        [x["MAPE"] for x in macro_seed_results],
        dtype=float,
    )
    rmse_values = np.array(
        [x["RMSE"] for x in macro_seed_results],
        dtype=float,
    )
    mae_values = np.array(
        [x["MAE"] for x in macro_seed_results],
        dtype=float,
    )
    r2_values = np.array(
        [x["R2"] for x in macro_seed_results],
        dtype=float,
    )

    ddof = 1 if len(macro_seed_results) > 1 else 0
    results["QRRFull_TinyMLP2"]["aggregate"]["Avg"] = {
        "accuracy_mean": round(float(acc_values.mean()), 6),
        "accuracy_std": round(float(acc_values.std(ddof=ddof)), 6),
        "MAPE_mean": round(float(mape_values.mean()), 8),
        "MAPE_std": round(float(mape_values.std(ddof=ddof)), 8),
        "RMSE_mean": round(float(rmse_values.mean()), 8),
        "RMSE_std": round(float(rmse_values.std(ddof=ddof)), 8),
        "MAE_mean": round(float(mae_values.mean()), 8),
        "MAE_std": round(float(mae_values.std(ddof=ddof)), 8),
        "R2_mean": round(float(r2_values.mean()), 8),
        "R2_std": round(float(r2_values.std(ddof=ddof)), 8),
        "n_seeds": len(macro_seed_results),
        "n_batches_per_seed": macro_seed_results[0]["n_batches"],
    }


# Save detailed artifacts.
with open(
    f"{RES_DIR}/XJTU_{MODEL_NAME}_TinyMLP2_10seed_results.json",
    "w",
) as f:
    json.dump(results, f, indent=2)

fold_df = pd.DataFrame(fold_rows)
fold_df.to_csv(
    f"{RES_DIR}/XJTU_{MODEL_NAME}_TinyMLP2_10seed_fold_results.csv",
    index=False,
)

seed_summary_df = pd.DataFrame(seed_summary_rows)
seed_summary_df.to_csv(
    f"{RES_DIR}/XJTU_{MODEL_NAME}_TinyMLP2_10seed_seed_summary.csv",
    index=False,
)


# Reference comparison table.
summary_rows = []
aggregate_results = results["QRRFull_TinyMLP2"]["aggregate"]

for bn in selected_batches:
    row = {"Batch": bn}
    for ref_name, ref_vals in REF_XJTU.items():
        row[f"Ref_{ref_name}"] = ref_vals.get(bn)

    batch_aggregate = aggregate_results.get(bn, {})
    row["New_QRRFull_TinyMLP2_mean"] = batch_aggregate.get("accuracy_mean")
    row["New_QRRFull_TinyMLP2_std"] = batch_aggregate.get("accuracy_std")
    summary_rows.append(row)

macro = {"Batch": "Avg"}
for ref_name, ref_vals in REF_XJTU.items():
    macro[f"Ref_{ref_name}"] = ref_vals.get("Avg")

macro_aggregate = aggregate_results.get("Avg", {})
macro["New_QRRFull_TinyMLP2_mean"] = macro_aggregate.get("accuracy_mean")
macro["New_QRRFull_TinyMLP2_std"] = macro_aggregate.get("accuracy_std")
summary_rows.append(macro)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(
    f"{RES_DIR}/XJTU_{MODEL_NAME}_TinyMLP2_10seed_summary.csv",
    index=False,
)

log(f"\n{'='*120}")
log("XJTU QRR_Full + TinyMLP2 — 10-seed summary")
log(f"{'='*120}")
print(summary_df.to_string(index=False))

if "Avg" in aggregate_results:
    avg_result = aggregate_results["Avg"]
    log(
        "FINAL 10-SEED MACRO → "
        f"MAPE={avg_result['MAPE_mean']:.6f}"
        f"±{avg_result['MAPE_std']:.6f} "
        f"RMSE={avg_result['RMSE_mean']:.6f}"
        f"±{avg_result['RMSE_std']:.6f} "
        f"Acc={avg_result['accuracy_mean']:.2f}"
        f"±{avg_result['accuracy_std']:.2f}%"
    )

log(f"✅ Done in {time.time()-t_global:.0f}s")
log(f"Results saved to: {RES_DIR}")


[06:34:13] 
  XJTU — QRR_Full features + TinyMLP2 readout — 10 seeds
[06:34:13]   Fixed QRR reservoir seed: 42
[06:34:13]   TinyMLP2 readout seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[06:34:13] 
----------------------------------------------------------------------
  Batch-1: 8 LOO folds × 10 readout seeds
----------------------------------------------------------------------
[06:34:14]     Fold 1/8: test=Batch-1_2C_battery-1
[06:34:14]       Generating QRR_Full features: train=2824, test=385
      500/2824 (38s, ~175s left)
      1000/2824 (68s, ~124s left)
      1500/2824 (89s, ~79s left)
      2000/2824 (112s, ~46s left)
      2500/2824 (133s, ~17s left)
      2824/2824 done in 146.0s                        
[06:39:17]       Seed 00 → MAPE=0.009149 RMSE=0.011020 Acc=99.09% params=125 iter=243 val_loss=0.000170
[06:39:35]       Seed 01 → MAPE=0.009311 RMSE=0.011545 Acc=99.07% params=125 iter=300 val_loss=0.000140
[06:39:49]       Seed 02 → MAPE=0.007066 RMSE=0.009100 Acc=99.29% params=12

## Notes

This notebook intentionally uses `REGENERATE_FEATURES = True` by default, so the logs prove that QRR_Full features were recomputed from XJTU data.

For a second readout-only rerun, change:

```python
REGENERATE_FEATURES = False
```

Do **not** change `MODEL_NAME`, `COUPLING_STRENGTH`, `WARMUP`, or TinyMLP2 settings if you want a strict comparison.

The ten-seed extension changes only the readout/validation seed. The QRR reservoir seed, recurrent-memory protocol, input features, LOO folds, normalization, warm-up, architecture, optimizer, learning rate, batch size, epochs, patience, weight decay, and all other scientific settings remain unchanged.
